# FireSight-IR | Module 3a — Multi-Branch Physics-Informed Neural Network

**Project:** FireSight-IR — FireSat Protoflight-aligned wildfire detection pipeline  
**Author:** Emmanuel Ibekwe | github.com/Ibekwemmanuel7  
**Module:** 3a of 9 — PINN architecture, training, validation  
**Platform:** Google Colab (GPU runtime required — Runtime → Change runtime type → T4 GPU)  
**Depends on:** Module 2 outputs in Google Drive

---

## Architecture overview

FireSight-IR uses a **multi-branch PINN** designed around the three-sensor data streams
from the FireSat Protoflight instrument:

```
Input streams:
  CNN branch   ← BT_I4, BT_I5, BTD, fire_mask patches (32×32)
  MLP-atm      ← ERA5 atmospheric state (16 features, normalised)
  MLP-srf      ← Surface + infrastructure context (20 features)
  MLP-derived  ← Derived physics features (6 features)

Fusion → Classification head → 3-class softmax
         (non-fire / wildfire / false-alarm)

Loss = CrossEntropy(weighted) + λ_BL * Beer-Lambert penalty
                               + λ_DR * Dynamic range penalty
                               + λ_TH * Thermal realism penalty
```

### CNN branch
- Input: 4-channel patch (BT_I4, BT_I5, BTD, fire_mask) at 32×32
- Architecture: 3× Conv2d → BatchNorm → ReLU → MaxPool, then GlobalAvgPool
- Output: 128-dim embedding

### MLP branches
- MLP-atm: 16 → 64 → 32 (2-layer residual)
- MLP-srf: 20 → 64 → 32 (2-layer residual)
- MLP-derived: 6 → 32 → 16 (2-layer)

### Fusion
- Concatenate: 128 + 32 + 32 + 16 = 208 dims
- Fusion MLP: 208 → 128 → 64 → 3
- Dropout 0.3 in fusion layers

### Physics loss terms

| Term | Formula | Motivation |
|---|---|---|
| Beer-Lambert | `MSE(pred_transmit, beer_lambert_proxy)` | IR signal must respect atmospheric attenuation |
| Dynamic range | `penalty if wildfire pred AND BT_I4 < 310K` | Real wildfires have elevated BT_I4 |
| Thermal realism | `penalty if wildfire pred AND BTD < 10K` | Fire BTD >> background BTD |

## Output

```
Google Drive: firesight-ir/
  models/
    firesight_pinn_best.pt       ← best val loss checkpoint
    firesight_pinn_final.pt      ← final epoch weights
    training_log.json            ← epoch-level metrics
  figures/
    03a_training_curves.png
    03a_confusion_matrix.png
    03a_roc_curves.png
```

---
## Section 0 — Setup

> **Required:** GPU runtime. Go to Runtime → Change runtime type → T4 GPU  
> This module will not run in reasonable time on CPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
!pip install torch torchvision tqdm scikit-learn matplotlib numpy pandas h5py pyarrow -q

import os, json, time, warnings
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR
import matplotlib.pyplot as plt
from sklearn.metrics import (confusion_matrix, classification_report,
                             roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from tqdm.notebook import tqdm
warnings.filterwarnings('ignore')

# Verify GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('[WARN] No GPU detected — training will be very slow')
    print('       Go to Runtime → Change runtime type → T4 GPU')

print(f'PyTorch: {torch.__version__}')

---
## Section 1 — Configuration

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = '/content/drive/MyDrive/firesight-ir'
ARCHIVE_PATH = f'{BASE_DIR}/data/processed/patches/firesight_patches.h5'
SCALERS_PATH = f'{BASE_DIR}/data/scalers/feature_scalers.json'
WEIGHTS_PATH = f'{BASE_DIR}/data/scalers/class_weights.json'
SPLIT_DIR    = f'{BASE_DIR}/data/splits'
MODEL_DIR    = f'{BASE_DIR}/models'
FIGURES_DIR  = f'{BASE_DIR}/figures'

for d in [MODEL_DIR, FIGURES_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Feature dimensions (must match Module 2 output) ──────────────────────────
N_ATM     = 16   # ERA5 features
N_SRF     = 20   # surface + infrastructure
N_DERIVED =  6   # Beer-Lambert, lifted index, doy, BT anomalies
PATCH_SIZE = 32
N_CNN_CHANNELS = 4  # BT_I4, BT_I5, BTD, fire_mask
N_CLASSES  = 3

# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE  = 512
N_EPOCHS    = 30
LR_MAX      = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT     = 0.3

# ── Physics loss weights ──────────────────────────────────────────────────────
# These scale the contribution of each physics constraint relative to CE loss
LAMBDA_BL   = 0.1   # Beer-Lambert transmittance consistency
LAMBDA_DR   = 0.05  # Dynamic range (wildfire must have elevated BT)
LAMBDA_TH   = 0.05  # Thermal realism (wildfire must have elevated BTD)

# Physics thresholds (from FireSat sensor specs and fire physics literature)
BT_I4_FIRE_MIN  = 310.0   # K — minimum credible fire BT at 3.7µm
BTD_FIRE_MIN    =  10.0   # K — minimum credible BTD for confirmed fire

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if device.type == 'cuda':
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

print('Configuration set.')
print(f'  Batch size : {BATCH_SIZE}')
print(f'  Epochs     : {N_EPOCHS}')
print(f'  Device     : {device}')
print(f'  Physics λ  : BL={LAMBDA_BL}, DR={LAMBDA_DR}, TH={LAMBDA_TH}')

---
## Section 2 — Dataset class

In [ ]:
class FireSightDataset(Dataset):
    """
    Lazy-loading HDF5 dataset for the FireSight-IR archive.

    Opens the HDF5 file once per worker (via _open_hdf5) to avoid
    serialisation issues with multiprocessing DataLoader.

    Each sample returns:
      patch      : (4, 32, 32) float32  — CNN input
      atm        : (16,)       float32  — MLP-atm input
      srf        : (20,)       float32  — MLP-srf input
      derived    : (6,)        float32  — MLP-derived input
      label      : int                  — 0/1/2
      physics_aux: (3,)        float32  — [BT_I4_center, BTD_center, beer_lambert]
                                           used to compute physics loss terms
    """

    def __init__(self, archive_path, indices, augment=False):
        self.archive_path = archive_path
        self.indices      = np.sort(indices)   # sorted for HDF5 efficiency
        self.augment      = augment
        self._hf          = None

        # Cache labels for class weighting (small enough to hold in memory)
        with h5py.File(archive_path, 'r') as hf:
            self.labels_cache = hf['labels'][self.indices]
            # Check which derived columns exist
            self.has_derived = 'mlp_derived' in hf

    def _open_hdf5(self):
        """Open HDF5 lazily — called in __getitem__ for multiprocess safety."""
        if self._hf is None:
            self._hf = h5py.File(self.archive_path, 'r')

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        self._open_hdf5()
        idx = int(self.indices[i])

        # CNN input: stack 4 channels
        bt_i4   = self._hf['cnn/bt_i4'][idx]       # (32, 32)
        bt_i5   = self._hf['cnn/bt_i5'][idx]       # (32, 32)
        bt_diff = self._hf['cnn/bt_diff'][idx]     # (32, 32)
        fm      = self._hf['cnn/fire_mask'][idx].astype(np.float32)  # (32, 32)

        patch = np.stack([bt_i4, bt_i5, bt_diff, fm], axis=0)  # (4, 32, 32)

        # Normalise BT channels to roughly [-1, 1] range
        # BT_I4 typical fire range ~280-450 K — centre at 300, scale by 50
        patch[0] = (patch[0] - 300.0) / 50.0
        patch[1] = (patch[1] - 290.0) / 20.0
        patch[2] = patch[2] / 40.0
        patch[3] = patch[3] / 9.0   # fire mask 0-9

        # Data augmentation (training only)
        if self.augment:
            k = np.random.randint(0, 4)
            patch = np.rot90(patch, k, axes=(1, 2)).copy()
            if np.random.rand() > 0.5:
                patch = np.flip(patch, axis=2).copy()

        # MLP inputs
        atm = self._hf['mlp_atm'][idx].astype(np.float32)
        srf = self._hf['mlp_srf'][idx].astype(np.float32)

        if self.has_derived:
            derived = self._hf['mlp_derived'][idx].astype(np.float32)
        else:
            derived = np.zeros(N_DERIVED, dtype=np.float32)

        # Replace NaN with 0 (safety)
        atm     = np.nan_to_num(atm,     nan=0.0)
        srf     = np.nan_to_num(srf,     nan=0.0)
        derived = np.nan_to_num(derived, nan=0.0)

        label = int(self._hf['labels'][idx])

        # Physics auxiliary values for loss computation
        # [BT_I4 center pixel, BTD center pixel, beer_lambert_proxy]
        bt_i4_center = float(self._hf['cnn/bt_i4'][idx, 16, 16])
        btd_center   = float(self._hf['cnn/bt_diff'][idx, 16, 16])
        # beer_lambert is index 14 in ATM vector (from Module 2 config)
        beer_lambert = float(atm[14]) if len(atm) > 14 else 0.72
        physics_aux  = np.array([bt_i4_center, btd_center, beer_lambert], dtype=np.float32)

        return (
            torch.from_numpy(patch.astype(np.float32)),
            torch.from_numpy(atm),
            torch.from_numpy(srf),
            torch.from_numpy(derived),
            torch.tensor(label, dtype=torch.long),
            torch.from_numpy(physics_aux),
        )

    def __del__(self):
        if self._hf is not None:
            try:
                self._hf.close()
            except Exception:
                pass


print('FireSightDataset defined.')

---
## Section 3 — Model architecture

In [ ]:
class ResidualBlock(nn.Module):
    """2-layer residual block for MLP branches."""
    def __init__(self, in_dim, out_dim, dropout=0.2):
        super().__init__()
        self.fc1  = nn.Linear(in_dim,  out_dim)
        self.fc2  = nn.Linear(out_dim, out_dim)
        self.bn1  = nn.BatchNorm1d(out_dim)
        self.bn2  = nn.BatchNorm1d(out_dim)
        self.drop = nn.Dropout(dropout)
        self.proj = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()

    def forward(self, x):
        identity = self.proj(x)
        out = F.relu(self.bn1(self.fc1(x)))
        out = self.drop(out)
        out = self.bn2(self.fc2(out))
        return F.relu(out + identity)


class CNNBranch(nn.Module):
    """
    Convolutional branch for 32×32 BT patches.
    Input: (B, 4, 32, 32)
    Output: (B, 128)
    """
    def __init__(self, in_channels=4, dropout=0.2):
        super().__init__()
        self.encoder = nn.Sequential(
            # Block 1: 32×32 → 16×16
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            # Block 2: 16×16 → 8×8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout2d(0.1),

            # Block 3: 8×8 → 4×4
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        # Global average pooling: 128 × 4 × 4 → 128
        self.gap  = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        x = self.encoder(x)
        x = self.gap(x).flatten(1)
        return self.drop(x)


class FireSightPINN(nn.Module):
    """
    Multi-branch Physics-Informed Neural Network for FireSight-IR.

    Branches:
      CNN     : 4-channel BT patches → 128-dim embedding
      MLP-atm : ERA5 atmospheric state → 32-dim embedding
      MLP-srf : Surface + infrastructure → 32-dim embedding
      MLP-der : Derived physics features → 16-dim embedding

    Fusion: concat(128+32+32+16=208) → 128 → 64 → 3

    Extra output: transmittance prediction (for Beer-Lambert physics loss)
    """

    def __init__(self, n_atm=16, n_srf=20, n_derived=6,
                 n_classes=3, dropout=0.3):
        super().__init__()

        # CNN branch
        self.cnn_branch = CNNBranch(in_channels=4, dropout=dropout)
        cnn_dim = 128

        # MLP-atm branch
        self.atm_branch = nn.Sequential(
            ResidualBlock(n_atm, 64, dropout=0.2),
            ResidualBlock(64,    32, dropout=0.2),
        )
        atm_dim = 32

        # MLP-srf branch
        self.srf_branch = nn.Sequential(
            ResidualBlock(n_srf, 64, dropout=0.2),
            ResidualBlock(64,    32, dropout=0.2),
        )
        srf_dim = 32

        # MLP-derived branch
        self.derived_branch = nn.Sequential(
            nn.Linear(n_derived, 32),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 16),
            nn.BatchNorm1d(16),
            nn.ReLU(),
        )
        derived_dim = 16

        # Fusion head
        fusion_in = cnn_dim + atm_dim + srf_dim + derived_dim  # 208

        self.fusion = nn.Sequential(
            nn.Linear(fusion_in, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.classifier = nn.Linear(64, n_classes)

        # Physics head: predict atmospheric transmittance
        # Used to compute Beer-Lambert consistency loss
        self.transmittance_head = nn.Sequential(
            nn.Linear(64, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid(),   # transmittance ∈ (0, 1)
        )

    def forward(self, patch, atm, srf, derived):
        """
        Returns:
          logits        : (B, 3)
          transmittance : (B, 1) — predicted IR transmittance
        """
        f_cnn     = self.cnn_branch(patch)
        f_atm     = self.atm_branch(atm)
        f_srf     = self.srf_branch(srf)
        f_derived = self.derived_branch(derived)

        fused    = torch.cat([f_cnn, f_atm, f_srf, f_derived], dim=1)
        features = self.fusion(fused)

        logits          = self.classifier(features)
        transmittance   = self.transmittance_head(features)

        return logits, transmittance


# Quick architecture check
model = FireSightPINN(N_ATM, N_SRF, N_DERIVED, N_CLASSES, DROPOUT).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'FireSightPINN architecture:')
print(f'  Total parameters    : {total_params:,}')
print(f'  Trainable parameters: {trainable:,}')

# Dry run
with torch.no_grad():
    dummy_patch   = torch.randn(4, N_CNN_CHANNELS, PATCH_SIZE, PATCH_SIZE).to(device)
    dummy_atm     = torch.randn(4, N_ATM).to(device)
    dummy_srf     = torch.randn(4, N_SRF).to(device)
    dummy_derived = torch.randn(4, N_DERIVED).to(device)
    logits, trans = model(dummy_patch, dummy_atm, dummy_srf, dummy_derived)
    print(f'  Dry run passed: logits={logits.shape}, transmittance={trans.shape}')

---
## Section 4 — Physics-informed loss function

In [ ]:
class FireSightPINNLoss(nn.Module):
    """
    Combined loss for FireSight-IR PINN.

    Total loss = L_CE + λ_BL * L_BeerLambert
                      + λ_DR * L_DynamicRange
                      + λ_TH * L_ThermalRealism

    L_CE          : Weighted cross-entropy (class weights from Module 2)
    L_BeerLambert : MSE between predicted transmittance and beer_lambert_proxy
                    from ERA5 TCWV. Enforces atmospheric physics consistency.
    L_DynamicRange: Penalty when model predicts wildfire (class 1) for pixels
                    with BT_I4 < BT_I4_FIRE_MIN (310 K). Low BT = not fire.
    L_ThermalRealism: Penalty when model predicts wildfire for pixels
                    with BTD < BTD_FIRE_MIN (10 K). Low BTD = likely not fire.

    Physical motivation:
    - Beer-Lambert: the atmosphere attenuates the MWIR signal before reaching
      the satellite. A model that ignores this will overestimate FRP for
      humid conditions and underestimate for dry conditions.
    - Dynamic range / Thermal realism: motivated directly by the FireSat
      Protoflight challenge — distinguishing gas flares (high BT, moderate BTD)
      from wildfires (very high BTD) and urban heat (low BTD, high radiance).
    """

    def __init__(self, class_weights, lambda_bl=0.1, lambda_dr=0.05,
                 lambda_th=0.05, bt_i4_fire_min=310.0, btd_fire_min=10.0,
                 device='cpu'):
        super().__init__()
        self.lambda_bl  = lambda_bl
        self.lambda_dr  = lambda_dr
        self.lambda_th  = lambda_th
        self.bt_min     = bt_i4_fire_min
        self.btd_min    = btd_fire_min

        weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
        self.ce_loss = nn.CrossEntropyLoss(weight=weights)
        self.mse     = nn.MSELoss()

    def forward(self, logits, transmittance, labels, physics_aux):
        """
        Parameters
        ----------
        logits        : (B, 3)  — model class logits
        transmittance : (B, 1)  — model predicted transmittance
        labels        : (B,)    — ground truth labels 0/1/2
        physics_aux   : (B, 3)  — [BT_I4_center, BTD_center, beer_lambert_proxy]
        """
        # ── Classification loss ───────────────────────────────────────────────
        L_ce = self.ce_loss(logits, labels)

        # ── Beer-Lambert transmittance consistency ────────────────────────────
        beer_lambert_target = physics_aux[:, 2:3]   # (B, 1)
        L_bl = self.mse(transmittance, beer_lambert_target)

        # ── Dynamic range penalty ─────────────────────────────────────────────
        # Predicted probability of wildfire
        probs       = F.softmax(logits, dim=1)
        p_wildfire  = probs[:, 1]                   # (B,)

        bt_i4   = physics_aux[:, 0]                 # (B,)
        # Penalise if model predicts high wildfire probability AND BT is too low
        low_bt_mask = (bt_i4 < self.bt_min).float()
        L_dr = (p_wildfire * low_bt_mask).mean()

        # ── Thermal realism penalty ───────────────────────────────────────────
        btd     = physics_aux[:, 1]                 # (B,)
        low_btd_mask = (btd < self.btd_min).float()
        L_th = (p_wildfire * low_btd_mask).mean()

        # ── Total loss ────────────────────────────────────────────────────────
        total = (L_ce
                 + self.lambda_bl * L_bl
                 + self.lambda_dr * L_dr
                 + self.lambda_th * L_th)

        return total, {
            'ce':  L_ce.item(),
            'bl':  L_bl.item(),
            'dr':  L_dr.item(),
            'th':  L_th.item(),
        }


print('FireSightPINNLoss defined.')

---
## Section 5 — Load splits, class weights and build DataLoaders

In [ ]:
# ── Load split indices ────────────────────────────────────────────────────────
train_idx = np.load(f'{SPLIT_DIR}/train_index.npy')
val_idx   = np.load(f'{SPLIT_DIR}/val_index.npy')
test_idx  = np.load(f'{SPLIT_DIR}/test_index.npy')

print(f'Split sizes — Train: {len(train_idx):,} | Val: {len(val_idx):,} | Test: {len(test_idx):,}')

# ── Load class weights from Module 2 ─────────────────────────────────────────
with open(WEIGHTS_PATH) as f:
    cw_data = json.load(f)

# Handle both list and dict formats from Module 2
if isinstance(cw_data, list):
    class_weights = cw_data
elif isinstance(cw_data, dict):
    class_weights = [cw_data.get('0', 18.98), cw_data.get('1', 1.0), cw_data.get('2', 27.02)]
else:
    class_weights = [18.98, 1.0, 27.02]  # Module 2 computed values

print(f'Class weights: non-fire={class_weights[0]:.2f}, '
      f'wildfire={class_weights[1]:.2f}, false-alarm={class_weights[2]:.2f}')

# ── Build datasets ────────────────────────────────────────────────────────────
train_ds = FireSightDataset(ARCHIVE_PATH, train_idx, augment=True)
val_ds   = FireSightDataset(ARCHIVE_PATH, val_idx,   augment=False)
test_ds  = FireSightDataset(ARCHIVE_PATH, test_idx,  augment=False)

# ── Build DataLoaders ─────────────────────────────────────────────────────────
# num_workers=2 works well on Colab; set to 0 if you hit multiprocessing errors
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=(device.type=='cuda'), prefetch_factor=2,
    persistent_workers=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=2, pin_memory=(device.type=='cuda'),
    persistent_workers=True,
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
    num_workers=2, pin_memory=(device.type=='cuda'),
)

print(f'DataLoaders ready.')
print(f'  Train batches: {len(train_loader):,}')
print(f'  Val batches  : {len(val_loader):,}')
print(f'  Test batches : {len(test_loader):,}')

# Verify one batch
patch, atm, srf, derived, label, aux = next(iter(train_loader))
print(f'\nSample batch shapes:')
print(f'  patch   : {patch.shape}')
print(f'  atm     : {atm.shape}')
print(f'  srf     : {srf.shape}')
print(f'  derived : {derived.shape}')
print(f'  label   : {label.shape} | values: {label.unique()}')
print(f'  aux     : {aux.shape}')

---
## Section 6 — Training loop

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    loss_components = {'ce': 0, 'bl': 0, 'dr': 0, 'th': 0}
    correct = 0
    n_samples = 0

    for patch, atm, srf, derived, labels, aux in loader:
        patch   = patch.to(device,   non_blocking=True)
        atm     = atm.to(device,     non_blocking=True)
        srf     = srf.to(device,     non_blocking=True)
        derived = derived.to(device, non_blocking=True)
        labels  = labels.to(device,  non_blocking=True)
        aux     = aux.to(device,     non_blocking=True)

        optimizer.zero_grad()
        logits, transmittance = model(patch, atm, srf, derived)
        loss, components = criterion(logits, transmittance, labels, aux)
        loss.backward()

        # Gradient clipping prevents exploding gradients in early training
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * len(labels)
        for k in loss_components:
            loss_components[k] += components[k] * len(labels)
        correct   += (logits.argmax(1) == labels).sum().item()
        n_samples += len(labels)

    avg_loss = total_loss / n_samples
    avg_acc  = correct / n_samples
    avg_components = {k: v / n_samples for k, v in loss_components.items()}
    return avg_loss, avg_acc, avg_components


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []
    all_probs  = []
    n_samples  = 0

    for patch, atm, srf, derived, labels, aux in loader:
        patch   = patch.to(device,   non_blocking=True)
        atm     = atm.to(device,     non_blocking=True)
        srf     = srf.to(device,     non_blocking=True)
        derived = derived.to(device, non_blocking=True)
        labels  = labels.to(device,  non_blocking=True)
        aux     = aux.to(device,     non_blocking=True)

        logits, transmittance = model(patch, atm, srf, derived)
        loss, _ = criterion(logits, transmittance, labels, aux)

        probs = F.softmax(logits, dim=1)
        preds = logits.argmax(1)

        total_loss += loss.item() * len(labels)
        all_preds.append(preds.cpu().numpy())
        all_labels.append(labels.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        n_samples  += len(labels)

    avg_loss  = total_loss / n_samples
    all_preds  = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    all_probs  = np.concatenate(all_probs, axis=0)
    acc = (all_preds == all_labels).mean()

    return avg_loss, acc, all_preds, all_labels, all_probs


print('Training loop functions defined.')

In [ ]:
# ── Initialise model, optimizer, scheduler, loss ──────────────────────────────
model     = FireSightPINN(N_ATM, N_SRF, N_DERIVED, N_CLASSES, DROPOUT).to(device)
optimizer = AdamW(model.parameters(), lr=LR_MAX, weight_decay=WEIGHT_DECAY)

# OneCycleLR: cosine annealing with warm-up — well-suited for PINN training
scheduler = OneCycleLR(
    optimizer,
    max_lr       = LR_MAX,
    steps_per_epoch = len(train_loader),
    epochs       = N_EPOCHS,
    pct_start    = 0.1,
    anneal_strategy = 'cos',
)

criterion = FireSightPINNLoss(
    class_weights = class_weights,
    lambda_bl     = LAMBDA_BL,
    lambda_dr     = LAMBDA_DR,
    lambda_th     = LAMBDA_TH,
    bt_i4_fire_min = BT_I4_FIRE_MIN,
    btd_fire_min   = BTD_FIRE_MIN,
    device         = device,
)

print('Optimizer and loss ready.')
print(f'  Optimizer   : AdamW (lr={LR_MAX}, wd={WEIGHT_DECAY})')
print(f'  Scheduler   : OneCycleLR ({N_EPOCHS} epochs)')
print(f'  Loss        : CE + BL + DR + TH')

In [ ]:
# ── Training run ──────────────────────────────────────────────────────────────
# Resume from checkpoint if it exists
BEST_MODEL_PATH  = f'{MODEL_DIR}/firesight_pinn_best.pt'
FINAL_MODEL_PATH = f'{MODEL_DIR}/firesight_pinn_final.pt'
LOG_PATH         = f'{MODEL_DIR}/training_log.json'

best_val_loss = float('inf')
training_log  = []
start_epoch   = 0

# Resume if checkpoint exists
if os.path.exists(BEST_MODEL_PATH):
    print(f'Resuming from checkpoint: {BEST_MODEL_PATH}')
    ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    best_val_loss = ckpt.get('val_loss', float('inf'))
    start_epoch   = ckpt.get('epoch', 0)
    if os.path.exists(LOG_PATH):
        with open(LOG_PATH) as f:
            training_log = json.load(f)
    print(f'  Resumed from epoch {start_epoch} | best val loss {best_val_loss:.4f}')

print(f'\nStarting training: epochs {start_epoch+1}–{N_EPOCHS}')
print(f'  Train batches/epoch: {len(train_loader):,}')
print()

for epoch in range(start_epoch, N_EPOCHS):
    t0 = time.time()

    # Train
    train_loss, train_acc, train_components = train_epoch(
        model, train_loader, optimizer, scheduler, criterion, device
    )

    # Validate
    val_loss, val_acc, val_preds, val_labels, val_probs = eval_epoch(
        model, val_loader, criterion, device
    )

    elapsed = time.time() - t0
    lr_now  = scheduler.get_last_lr()[0]

    # Compute per-class accuracy on val set
    per_class_acc = {}
    for cls, name in enumerate(['non-fire', 'wildfire', 'false-alarm']):
        mask = (val_labels == cls)
        if mask.sum() > 0:
            per_class_acc[name] = (val_preds[mask] == cls).mean()

    # Log
    epoch_log = {
        'epoch'      : epoch + 1,
        'train_loss' : round(train_loss, 4),
        'train_acc'  : round(train_acc,  4),
        'val_loss'   : round(val_loss,   4),
        'val_acc'    : round(val_acc,    4),
        'lr'         : round(lr_now,     8),
        'elapsed_s'  : round(elapsed,    1),
        'loss_components': {k: round(v, 5) for k, v in train_components.items()},
        'per_class_val_acc': {k: round(v, 4) for k, v in per_class_acc.items()},
    }
    training_log.append(epoch_log)

    print(f'Epoch {epoch+1:3d}/{N_EPOCHS} | '
          f'train_loss={train_loss:.4f} acc={train_acc:.3f} | '
          f'val_loss={val_loss:.4f} acc={val_acc:.3f} | '
          f'lr={lr_now:.2e} | {elapsed:.0f}s')
    print(f'  Loss: CE={train_components["ce"]:.4f} '
          f'BL={train_components["bl"]:.4f} '
          f'DR={train_components["dr"]:.4f} '
          f'TH={train_components["th"]:.4f}')
    if per_class_acc:
        nf = per_class_acc.get('non-fire', 0)
        wf = per_class_acc.get('wildfire', 0)
        fa = per_class_acc.get('false-alarm', 0)
        print(f'  Val per-class: non-fire={nf:.3f} wildfire={wf:.3f} false-alarm={fa:.3f}')

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({
            'epoch'            : epoch + 1,
            'model_state_dict' : model.state_dict(),
            'val_loss'         : val_loss,
            'val_acc'          : val_acc,
            'class_weights'    : class_weights,
        }, BEST_MODEL_PATH)
        print(f'  ✓ Best model saved (val_loss={val_loss:.4f})')

    # Save training log every epoch
    with open(LOG_PATH, 'w') as f:
        json.dump(training_log, f, indent=2)

# Save final model
torch.save({
    'epoch'            : N_EPOCHS,
    'model_state_dict' : model.state_dict(),
    'val_loss'         : val_loss,
}, FINAL_MODEL_PATH)

print(f'\n✓ Training complete')
print(f'  Best val loss : {best_val_loss:.4f}')
print(f'  Models saved  → {MODEL_DIR}')

---
## Section 7 — Training curves

In [ ]:
with open(LOG_PATH) as f:
    log = json.load(f)

epochs      = [e['epoch']      for e in log]
train_losses = [e['train_loss'] for e in log]
val_losses   = [e['val_loss']   for e in log]
train_accs   = [e['train_acc']  for e in log]
val_accs     = [e['val_acc']    for e in log]
ce_losses    = [e['loss_components']['ce'] for e in log]
bl_losses    = [e['loss_components']['bl'] for e in log]
dr_losses    = [e['loss_components']['dr'] for e in log]
th_losses    = [e['loss_components']['th'] for e in log]

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0a0a0a')
fig.patch.set_facecolor('#0a0a0a')

# Total loss
ax = axes[0]
ax.set_facecolor('#0a0a0a')
ax.plot(epochs, train_losses, color='#4FC3F7', label='Train')
ax.plot(epochs, val_losses,   color='#FF7043', label='Val (2023)')
ax.set_title('Total Loss', color='white', fontsize=11)
ax.set_xlabel('Epoch', color='#888')
ax.tick_params(colors='#666')
ax.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=9)
for sp in ax.spines.values(): sp.set_edgecolor('#333')

# Accuracy
ax = axes[1]
ax.set_facecolor('#0a0a0a')
ax.plot(epochs, train_accs, color='#4FC3F7', label='Train')
ax.plot(epochs, val_accs,   color='#FF7043', label='Val (2023)')
ax.set_title('Accuracy', color='white', fontsize=11)
ax.set_xlabel('Epoch', color='#888')
ax.set_ylim(0, 1)
ax.tick_params(colors='#666')
ax.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=9)
for sp in ax.spines.values(): sp.set_edgecolor('#333')

# Physics loss decomposition
ax = axes[2]
ax.set_facecolor('#0a0a0a')
ax.plot(epochs, ce_losses, color='#FFD700', label='CE')
ax.plot(epochs, bl_losses, color='#4CAF50', label='Beer-Lambert')
ax.plot(epochs, dr_losses, color='#FF69B4', label='Dynamic range')
ax.plot(epochs, th_losses, color='#00BCD4', label='Thermal realism')
ax.set_title('Physics Loss Decomposition', color='white', fontsize=11)
ax.set_xlabel('Epoch', color='#888')
ax.tick_params(colors='#666')
ax.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
for sp in ax.spines.values(): sp.set_edgecolor('#333')

fig.suptitle('FireSight-IR | Module 3a — Training Curves',
             color='white', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03a_training_curves.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('Training curves saved.')

---
## Section 8 — Evaluation on test set

In [ ]:
# Load best checkpoint for evaluation
ckpt = torch.load(BEST_MODEL_PATH, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]} '
      f'(val_loss={ckpt["val_loss"]:.4f})')

# Evaluate on test set (2018-2022 held-out 20%)
test_loss, test_acc, test_preds, test_labels, test_probs = eval_epoch(
    model, test_loader, criterion, device
)

print(f'\n=== Test Set Results (2018-2022 held-out) ===')
print(f'  Loss    : {test_loss:.4f}')
print(f'  Accuracy: {test_acc:.4f}')
print()
print(classification_report(
    test_labels, test_preds,
    target_names=['non-fire', 'wildfire', 'false-alarm'],
    digits=4
))

# Evaluate on val set (2023 held-out)
val_loss, val_acc, val_preds, val_labels, val_probs = eval_epoch(
    model, val_loader, criterion, device
)

print(f'=== Validation Set Results (2023 — fully held out) ===')
print(f'  Loss    : {val_loss:.4f}')
print(f'  Accuracy: {val_acc:.4f}')
print()
print(classification_report(
    val_labels, val_preds,
    target_names=['non-fire', 'wildfire', 'false-alarm'],
    digits=4
))

In [ ]:
# ── Confusion matrices ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#0a0a0a')
fig.patch.set_facecolor('#0a0a0a')
class_names = ['Non-fire', 'Wildfire', 'False-alarm']

for ax, preds, labels, title in [
    (axes[0], test_preds, test_labels, 'Test set (2018-2022)'),
    (axes[1], val_preds,  val_labels,  'Val set (2023)'),
]:
    cm = confusion_matrix(labels, preds, normalize='true')
    im = ax.imshow(cm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(3))
    ax.set_yticks(range(3))
    ax.set_xticklabels(class_names, rotation=30, ha='right', color='white', fontsize=9)
    ax.set_yticklabels(class_names, color='white', fontsize=9)
    ax.set_xlabel('Predicted', color='#888', fontsize=10)
    ax.set_ylabel('True',      color='#888', fontsize=10)
    ax.set_title(title, color='white', fontsize=11)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, f'{cm[i,j]:.3f}', ha='center', va='center',
                    color='white' if cm[i,j] < 0.5 else 'black', fontsize=10)
    plt.colorbar(im, ax=ax, fraction=0.046)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')

fig.suptitle('FireSight-IR | Module 3a — Confusion Matrices (normalised)',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03a_confusion_matrix.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()

In [ ]:
# ── ROC curves ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6), facecolor='#0a0a0a')
fig.patch.set_facecolor('#0a0a0a')
colors_roc = ['#FFD700', '#FF4500', '#00BCD4']

for ax, probs, labels, title in [
    (axes[0], test_probs, test_labels, 'Test set (2018-2022)'),
    (axes[1], val_probs,  val_labels,  'Val set (2023)'),
]:
    ax.set_facecolor('#0a0a0a')
    ax.plot([0,1],[0,1], 'k--', alpha=0.3)

    for cls_idx, (cls_name, color) in enumerate(
        zip(['non-fire', 'wildfire', 'false-alarm'], colors_roc)
    ):
        binary_labels = (labels == cls_idx).astype(int)
        if binary_labels.sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(binary_labels, probs[:, cls_idx])
        auc = roc_auc_score(binary_labels, probs[:, cls_idx])
        ax.plot(fpr, tpr, color=color, label=f'{cls_name} AUC={auc:.4f}')

    ax.set_xlabel('False Positive Rate', color='#888')
    ax.set_ylabel('True Positive Rate', color='#888')
    ax.set_title(title, color='white', fontsize=11)
    ax.tick_params(colors='#666')
    ax.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=9)
    for sp in ax.spines.values(): sp.set_edgecolor('#333')

fig.suptitle('FireSight-IR | Module 3a — ROC Curves (one-vs-rest)',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03a_roc_curves.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('ROC curves saved.')

---
## Section 9 — Physics constraint analysis

This section verifies that the physics loss terms are working as intended:
- Beer-Lambert: model's predicted transmittance should correlate with ERA5 TCWV
- Dynamic range: model should rarely predict wildfire for low-BT pixels
- Thermal realism: model should rarely predict wildfire for low-BTD pixels

In [ ]:
@torch.no_grad()
def collect_physics_outputs(model, loader, device, max_samples=50000):
    """Collect predicted transmittances and physics auxiliary values for analysis."""
    model.eval()
    all_trans  = []
    all_aux    = []
    all_preds  = []
    all_labels = []
    n = 0

    for patch, atm, srf, derived, labels, aux in loader:
        if n >= max_samples:
            break
        patch   = patch.to(device)
        atm     = atm.to(device)
        srf     = srf.to(device)
        derived = derived.to(device)

        logits, transmittance = model(patch, atm, srf, derived)
        all_trans.append(transmittance.cpu().numpy())
        all_aux.append(aux.numpy())
        all_preds.append(logits.argmax(1).cpu().numpy())
        all_labels.append(labels.numpy())
        n += len(labels)

    return (
        np.concatenate(all_trans)[:max_samples],
        np.concatenate(all_aux)[:max_samples],
        np.concatenate(all_preds)[:max_samples],
        np.concatenate(all_labels)[:max_samples],
    )

print('Collecting physics outputs from val set...')
trans, aux_vals, preds, labels_val = collect_physics_outputs(
    model, val_loader, device, max_samples=50000
)

bt_i4_vals    = aux_vals[:, 0]
btd_vals      = aux_vals[:, 1]
beer_targets  = aux_vals[:, 2]
trans_flat    = trans.flatten()

fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0a0a0a')
fig.patch.set_facecolor('#0a0a0a')

# Beer-Lambert: predicted vs target transmittance
ax = axes[0]
ax.set_facecolor('#0a0a0a')
ax.scatter(beer_targets[:5000], trans_flat[:5000], alpha=0.2, s=1,
           color='#4CAF50')
ax.plot([0, 1], [0, 1], 'r--', alpha=0.5)
corr = np.corrcoef(beer_targets, trans_flat)[0, 1]
ax.set_title(f'Beer-Lambert\n(pred vs target, r={corr:.3f})',
             color='white', fontsize=10)
ax.set_xlabel('ERA5 TCWV transmittance proxy', color='#888')
ax.set_ylabel('Model predicted transmittance', color='#888')
ax.tick_params(colors='#666')
for sp in ax.spines.values(): sp.set_edgecolor('#333')

# Dynamic range: BT_I4 distribution by predicted class
ax = axes[1]
ax.set_facecolor('#0a0a0a')
colors_dr = ['#FFD700', '#FF4500', '#00BCD4']
for cls, (name, color) in enumerate(zip(['non-fire', 'wildfire', 'false-alarm'], colors_dr)):
    vals = bt_i4_vals[preds == cls]
    if len(vals) > 0:
        ax.hist(vals, bins=50, alpha=0.5, color=color, label=name,
                density=True)
ax.axvline(BT_I4_FIRE_MIN, color='white', linestyle='--', alpha=0.5,
           label=f'Min fire BT ({BT_I4_FIRE_MIN}K)')
ax.set_title('BT_I4 by predicted class\n(dynamic range check)',
             color='white', fontsize=10)
ax.set_xlabel('BT_I4 center pixel (K)', color='#888')
ax.tick_params(colors='#666')
ax.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
for sp in ax.spines.values(): sp.set_edgecolor('#333')

# Thermal realism: BTD distribution by predicted class
ax = axes[2]
ax.set_facecolor('#0a0a0a')
for cls, (name, color) in enumerate(zip(['non-fire', 'wildfire', 'false-alarm'], colors_dr)):
    vals = btd_vals[preds == cls]
    if len(vals) > 0:
        ax.hist(vals, bins=50, alpha=0.5, color=color, label=name,
                density=True)
ax.axvline(BTD_FIRE_MIN, color='white', linestyle='--', alpha=0.5,
           label=f'Min fire BTD ({BTD_FIRE_MIN}K)')
ax.set_title('BTD by predicted class\n(thermal realism check)',
             color='white', fontsize=10)
ax.set_xlabel('BTD center pixel (K)', color='#888')
ax.tick_params(colors='#666')
ax.legend(facecolor='#1a1a1a', labelcolor='white', fontsize=8)
for sp in ax.spines.values(): sp.set_edgecolor('#333')

fig.suptitle('FireSight-IR | Module 3a — Physics Constraint Analysis',
             color='white', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/03a_physics_analysis.png',
            dpi=150, bbox_inches='tight', facecolor='#0a0a0a')
plt.show()
print('Physics analysis saved.')

---
## Section 10 — Module 3a summary

In [ ]:
print('=' * 55)
print('  FireSight-IR | Module 3a Complete')
print('=' * 55)
print(f'  Architecture : Multi-branch PINN')
print(f'  Parameters   : {sum(p.numel() for p in model.parameters()):,}')
print(f'  Branches     : CNN (32×32) + MLP-atm + MLP-srf + MLP-derived')
print(f'  Physics loss : Beer-Lambert + Dynamic Range + Thermal Realism')
print(f'  Training     : {N_EPOCHS} epochs, OneCycleLR, AdamW')
print(f'  Best val loss: {best_val_loss:.4f}')
print(f'  Val accuracy : {val_acc:.4f} (2023 held-out)')
print()
print(f'  Outputs:')
print(f'    {BEST_MODEL_PATH}')
print(f'    {FINAL_MODEL_PATH}')
print(f'    {LOG_PATH}')
print()
print(f'  Next: Module 3b — False-alarm discriminator (Stage D)')
print('=' * 55)